In [1]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path, MINIO_SPARK_ENDPOINT

spark = create_spark_session(
    "NYC Building Risk - Silver HPD Test"
)

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_SPARK_ENDPOINT)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 19:10:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/01 19:10:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.4.0
MinIO endpoint: http://host.docker.internal:9001


In [2]:
bronze_file = minio_path(
    "bronze/hpd_violations/"
    "year=2026/"
    "month=08/"
    "day=18/"
    "backfill/"
    "page_00001.json"
)

print("Reading:")
print(bronze_file)

Reading:
s3a://nyc-building-risk/bronze/hpd_violations/year=2026/month=08/day=18/backfill/page_00001.json


In [3]:
hpd_df = (
    spark.read
    .option("multiline", "true")
    .json(bronze_file)
)

print("Rows:", hpd_df.count())

26/09/01 19:11:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Rows: 1000


In [4]:
hpd_df.printSchema()

root
 |-- apartment: string (nullable = true)
 |-- approveddate: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- block: string (nullable = true)
 |-- boro: string (nullable = true)
 |-- boroid: string (nullable = true)
 |-- buildingid: string (nullable = true)
 |-- censustract: string (nullable = true)
 |-- certifieddate: string (nullable = true)
 |-- class: string (nullable = true)
 |-- communityboard: string (nullable = true)
 |-- councildistrict: string (nullable = true)
 |-- currentstatus: string (nullable = true)
 |-- currentstatusdate: string (nullable = true)
 |-- currentstatusid: string (nullable = true)
 |-- highhousenumber: string (nullable = true)
 |-- housenumber: string (nullable = true)
 |-- inspectiondate: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- lowhousenumber: string (nullable = true)
 |-- novdescription: string (n

In [5]:
hpd_df.select(
    "violationid",
    "buildingid",
    "boro",
    "block",
    "lot",
    "class",
    "inspectiondate",
    "currentstatus",
    "violationstatus",
    "novdescription"
).show(
    10,
    truncate=False
)

+-----------+----------+-----+-----+---+-----+-----------------------+---------------------------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|violationid|buildingid|boro |block|lot|class|inspectiondate         |currentstatus                    |violationstatus|novdescription                                                                                                                                                                                                   |
+-----------+----------+-----+-----+---+-----+-----------------------+---------------------------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|191456

In [6]:
from pyspark.sql import functions as F

hpd_df.select(
    "violationid",
    "bbl",
    "bin",
    "boroid",
    "boro",
    "block",
    "lot"
).show(
    20,
    truncate=False
)

print(
    "Missing BBL:",
    hpd_df
    .filter(
        F.col("bbl").isNull()
        | (F.trim(F.col("bbl")) == "")
    )
    .count()
)

print(
    "Missing BIN:",
    hpd_df
    .filter(
        F.col("bin").isNull()
        | (F.trim(F.col("bin")) == "")
    )
    .count()
)

print(
    "Invalid BBL format:",
    hpd_df
    .filter(
        F.col("bbl").isNotNull()
        & ~F.trim(F.col("bbl")).rlike("^[0-9]{10}$")
    )
    .count()
)

print(
    "Missing violationid:",
    hpd_df
    .filter(
        F.col("violationid").isNull()
        | (F.trim(F.col("violationid")) == "")
    )
    .count()
)

print(
    "Duplicate violationid:",
    hpd_df.count()
    - hpd_df.select("violationid").distinct().count()
)

+-----------+----------+-------+------+---------+-----+---+
|violationid|bbl       |bin    |boroid|boro     |block|lot|
+-----------+----------+-------+------+---------+-----+---+
|19145691   |2033090023|2094743|2     |BRONX    |3309 |23 |
|19145692   |2033090023|2094743|2     |BRONX    |3309 |23 |
|19145693   |2033090023|2094743|2     |BRONX    |3309 |23 |
|19145694   |2033090023|2094743|2     |BRONX    |3309 |23 |
|19145695   |2043370064|2049729|2     |BRONX    |4337 |64 |
|19145696   |2043370064|2049729|2     |BRONX    |4337 |64 |
|19145697   |2043370064|2049729|2     |BRONX    |4337 |64 |
|19145698   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145699   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145700   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145701   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145702   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145703   |2032560048|2129755|2     |BRONX    |3256 |48 |
|19145704   |3013850023|3037113|3     |B

In [7]:
spark.stop()

In [8]:
spark.stop()

In [9]:
silver_hpd_path = minio_path(
    "silver/hpd"
)

hpd_silver_df = (
    spark.read
    .parquet(silver_hpd_path)
)

print(
    "HPD Silver rows:",
    hpd_silver_df.count()
)

Py4JJavaError: An error occurred while calling o96.parquet.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.w

In [10]:
hpd_silver_df.groupBy(
    "inspection_year",
    "inspection_month"
).count().orderBy(
    "inspection_year",
    "inspection_month"
).show(
    50,
    truncate=False
)

NameError: name 'hpd_silver_df' is not defined

In [11]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path

spark = create_spark_session(
    "NYC Building Risk - HPD Silver Validation"
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 3.4.0
Spark UI: http://ff7529cf1dd6:4041


26/09/01 19:43:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [12]:
silver_hpd_path = minio_path(
    "silver/hpd"
)

hpd_silver_df = (
    spark.read
    .parquet(silver_hpd_path)
)

print(
    "HPD Silver rows:",
    hpd_silver_df.count()
)

HPD Silver rows: 927308


In [13]:
hpd_silver_df.groupBy(
    "inspection_year",
    "inspection_month"
).count().orderBy(
    "inspection_year",
    "inspection_month"
).show(
    50,
    truncate=False
)

+---------------+----------------+------+
|inspection_year|inspection_month|count |
+---------------+----------------+------+
|2025           |8               |14748 |
|2025           |9               |63361 |
|2025           |10              |70060 |
|2025           |11              |130685|
|2025           |12              |66506 |
|2026           |1               |63228 |
|2026           |2               |61286 |
|2026           |3               |72289 |
|2026           |4               |89668 |
|2026           |5               |119241|
|2026           |6               |69757 |
|2026           |7               |61692 |
|2026           |8               |44787 |
+---------------+----------------+------+



In [14]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path

spark = create_spark_session(
    "NYC Building Risk - Silver DOB Test"
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 3.4.0
Spark UI: http://ff7529cf1dd6:4041


26/09/01 19:46:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [15]:
bronze_file = minio_path(
    "bronze/dob_violations/"
    "year=2026/"
    "month=08/"
    "day=19/"
    "backfill/"
    "page_00001.json"
)

print("Reading:")
print(bronze_file)

Reading:
s3a://nyc-building-risk/bronze/dob_violations/year=2026/month=08/day=19/backfill/page_00001.json


In [16]:
dob_df.printSchema()

NameError: name 'dob_df' is not defined

In [17]:
bronze_file = minio_path(
    "bronze/dob_violations/"
    "year=2026/"
    "month=08/"
    "day=19/"
    "backfill/"
    "page_00001.json"
)

print("Reading:")
print(bronze_file)

Reading:
s3a://nyc-building-risk/bronze/dob_violations/year=2026/month=08/day=19/backfill/page_00001.json


In [18]:
dob_df = (
    spark.read
    .option("multiline", "true")
    .json(bronze_file)
)

print("Rows:", dob_df.count())

Rows: 134


In [19]:
dob_df.printSchema()

root
 |-- bbl: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- block: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- census_tract_2020_: string (nullable = true)
 |-- city: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- house_number: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- neighborhood_tabulation_area_nta_2020_: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street: string (nullable = true)
 |-- violation_issue_date: string (nullable = true)
 |-- violation_number: string (nullable = true)
 |-- violation_status: string (nullable = true)
 |-- violation_type: string (nullable = true)
 |-- zip: string (nullable = true)



In [20]:
from pyspark.sql import functions as F


dob_df.select(
    "violation_number",
    "violation_issue_date",
    "bbl",
    "bin",
    "borough",
    "block",
    "lot",
    "violation_type",
    "violation_status"
).show(
    20,
    truncate=False
)


print(
    "Missing violation_number:",
    dob_df
    .filter(
        F.col("violation_number").isNull()
        | (F.trim(F.col("violation_number")) == "")
    )
    .count()
)


print(
    "Duplicate violation_number:",
    dob_df.count()
    - dob_df.select("violation_number").distinct().count()
)


print(
    "Missing violation_issue_date:",
    dob_df
    .filter(
        F.col("violation_issue_date").isNull()
        | (F.trim(F.col("violation_issue_date")) == "")
    )
    .count()
)


print(
    "Missing BBL:",
    dob_df
    .filter(
        F.col("bbl").isNull()
        | (F.trim(F.col("bbl")) == "")
    )
    .count()
)


print(
    "Invalid BBL format:",
    dob_df
    .filter(
        F.col("bbl").isNotNull()
        & ~F.trim(F.col("bbl")).rlike("^[0-9]{10}$")
    )
    .count()
)


print(
    "Missing BIN:",
    dob_df
    .filter(
        F.col("bin").isNull()
        | (F.trim(F.col("bin")) == "")
    )
    .count()
)


print(
    "Missing coordinates:",
    dob_df
    .filter(
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    .count()
)

+------------------------------+-----------------------+----------+-------+---------+-----+---+--------------+----------------+
|violation_number              |violation_issue_date   |bbl       |bin    |borough  |block|lot|violation_type|violation_status|
+------------------------------+-----------------------+----------+-------+---------+-----+---+--------------+----------------+
|VIO-FTC-AEU-HAZ-202608-0039432|2026-08-19T00:00:00.000|3063460041|3165849|Brooklyn |6346 |41 |FTC-AEU-HAZ   |Active          |
|VIO-FTC-AEU-HAZ-202608-0039433|2026-08-19T00:00:00.000|1008020029|1015039|Manhattan|802  |29 |FTC-AEU-HAZ   |Active          |
|VIO-FTC-AEU-HAZ-202608-0039434|2026-08-19T00:00:00.000|1010270063|1024888|Manhattan|1027 |63 |FTC-AEU-HAZ   |Active          |
|VIO-FTC-AEU-HAZ-202608-0039435|2026-08-19T00:00:00.000|1010270063|1024888|Manhattan|1027 |63 |FTC-AEU-HAZ   |Active          |
|VIO-FTC-AEU-HAZ-202608-0039436|2026-08-19T00:00:00.000|1010270063|1024888|Manhattan|1027 |63 |FTC-AEU-H

In [21]:
spark.stop()